# How to use the MessagePack dataset

**IMPORTANT NOTE**
> I, the author of this notebook, eventually decided to implement the dataloader myself. It is then simpler for you to read the `dataLoader.ts` file in the `steaminghot/src` directory to understand how the dataset is loaded. However, this notebook explains what happens under the hood.

The message packet dataset is the main dataset used by our website. It is a binary file that contains the preprocessed dataset in a compact format. To use the dataset in the website (JavaScript) we can use the `msgpack` library, which is a JavaScript implementation of the MessagePack format. For this example notebook, we will use the equivalent Python code to show how to read the dataset, but the same principles apply in JavaScript.

## Loading the dataset

This part is straightforward.

In [7]:
import msgpack
import os
from typing import Any
from pathlib import Path

# STEAMINGHOT\Documentation\how-to-use-msgpack.ipynb
STEAMINGHOT_DIR = Path.cwd().parent
DATASET_PATH = os.path.join(STEAMINGHOT_DIR, "steaminghot/public/data/games.msgpack") 

compact = {}
with open(DATASET_PATH, "rb") as f:
    packed = f.read()
    compact = msgpack.unpackb(packed)
    if compact is None:
        raise ValueError("Failed to unpack dataset")
print(compact.keys())

dict_keys(['schema', 'games'])


The compact form of the dataset stores the data as follows:

``` JavaScript
{
    "schema": ['game_id', 'name', 'release_date', 'required_age',...],
    "games": [
        ['2539430', 'Black Dragon Mage Playtest', 'Aug 1, 2023', 0,...],
        ['496350', 'Supipara - Chapter 1 Spring Has Come!', 'Jul 29, 2016', 0,...],
        ....
    ]
}
```

## Mapping the schema to the game data

This step will allow us to work with the dataset in a more convenient way. The original format is compact and efficient for storage, but not very user-friendly for analysis. By mapping the schema to the game data, we can easily access the fields of each game using their names instead of indices. For example, we will be able to do `game['name']` instead of `game[schema.index('name')]`.

In [8]:
schema: list[str] = compact['schema']
dataset: dict[str, Any] = {}
for game_data in compact['games']:
    game_id = schema.index('game_id')
    dataset[game_data[game_id]] = {schema[i]: game_data[i] for i in range(len(schema))}

Now it is easier to work with the dataset:

In [9]:
outerwilds = dataset['753640']
print("Outer Wilds")
print('-'* 15)
print("release date:", outerwilds['release_date'])
print("short description:", outerwilds['short_description'])
print("positive reviews:", outerwilds['positive'])
print("negative reviews:", outerwilds['negative'], "(those are dumbdumbs)")


Outer Wilds
---------------
release date: Jun 18, 2020
short description: Named Game of the Year 2019 by Giant Bomb, Polygon, Eurogamer, and The Guardian, Outer Wilds is a critically-acclaimed and award-winning open world mystery about a solar system trapped in an endless time loop.
positive reviews: 80833
negative reviews: 3655 (those are dumbdumbs)


## Available fields in the dataset

Here a are all available fields for each game in the dataset:

In [10]:
for fields in schema:
    print(f"{fields}: {type(outerwilds[fields]).__name__}")

game_id: str
name: str
release_date: str
required_age: int
price: float
dlc_count: int
short_description: str
header_image: str
windows: bool
mac: bool
linux: bool
metacritic_score: int
metacritic_url: str
achievements: int
recommendations: int
notes: str
supported_languages: list
full_audio_languages: list
developers: list
publishers: list
categories: list
genres: list
movies: list
user_score: int
positive: int
negative: int
estimated_owners: str
average_playtime_forever: int
average_playtime_2weeks: int
median_playtime_forever: int
median_playtime_2weeks: int
discount: str
peak_ccu: int
tags: dict
